# 00 — Configuración inicial del entorno

**Proyecto:** Chem RAG Assistant  
**Repositorio:** https://github.com/Jesusrodriguezf90/chem-rag-assistant  
**Fase:** Configuración previa al pipeline

---

⚠️ **Este notebook se ejecuta una única vez** por entorno (Colab, servidor, local).  
Su función es preparar la estructura de carpetas y archivos de configuración  
necesarios antes de ejecutar cualquier otro notebook del proyecto.

In [ ]:
"""
Notebook: 00_setup.ipynb

Objetivo:
    Configurar el entorno de ejecución del proyecto Chem RAG Assistant.
    Crea la estructura de carpetas necesaria en Google Drive y genera
    los archivos de configuración del pipeline.

    Este notebook debe ejecutarse una única vez por entorno.
    No forma parte del pipeline de datos — es infraestructura.

Autor:   Jesús Rodríguez
Fecha:   2026-04-30
Versión: 1.0.0
"""

## 1. Configuración del entorno

In [1]:
# Detección del entorno de ejecución (Colab vs local)
try:
    from google.colab import drive
    IN_COLAB = True
    print("Entorno detectado: Google Colab")
except ImportError:
    IN_COLAB = False
    print("Entorno detectado: local")

Entorno detectado: Google Colab


In [2]:
# Montaje de Google Drive (solo en Colab)
if IN_COLAB:
    drive.mount('/content/drive')
    print("Google Drive montado correctamente")

Mounted at /content/drive
Google Drive montado correctamente


## 2. Definición de rutas

In [3]:
# Librería estándar para manejo de rutas
from pathlib import Path

# Ruta raíz del proyecto en Google Drive
PROYECTO_RAIZ = Path('/content/drive/MyDrive/chem-rag-assistant')

# Estructura de carpetas del proyecto
CARPETAS = [
    PROYECTO_RAIZ / 'data' / 'raw',
    PROYECTO_RAIZ / 'data' / 'processed',
    PROYECTO_RAIZ / 'data' / 'embeddings',
    PROYECTO_RAIZ / 'config',
    PROYECTO_RAIZ / 'notebooks',
]

print(f"Ruta raíz del proyecto: {PROYECTO_RAIZ}")

Ruta raíz del proyecto: /content/drive/MyDrive/chem-rag-assistant


## 3. Creación de la estructura de carpetas

In [4]:
# Creación de todas las carpetas necesarias
# exist_ok=True evita errores si la carpeta ya existe
print("Creando estructura de carpetas...")
print("-" * 45)

for carpeta in CARPETAS:
    carpeta.mkdir(parents=True, exist_ok=True)
    estado = "OK" if carpeta.exists() else "ERROR"
    print(f"  [{estado}]  {carpeta.relative_to(PROYECTO_RAIZ)}")

print("-" * 45)
print("Estructura de carpetas lista")

Creando estructura de carpetas...
---------------------------------------------
  [OK]  data/raw
  [OK]  data/processed
  [OK]  data/embeddings
  [OK]  config
  [OK]  notebooks
---------------------------------------------
Estructura de carpetas lista


## 4. Generación del archivo de configuración YAML

In [6]:
# El archivo cleaning_rules.yaml define las reglas de limpieza
# específicas por documento. La limpieza universal (placeholders,
# saltos de línea) se aplica siempre en 01_ingesta.ipynb.
# Este archivo solo es necesario para artefactos específicos
# de un publisher o formato concreto.
#
# Para añadir un nuevo documento: agregar una entrada nueva
# con el nombre del PDF (sin extensión) como clave.
# Si el documento no necesita limpieza específica, no es
# necesario añadirlo aquí.
import yaml

%pip install pyyaml -q

RUTA_YAML = PROYECTO_RAIZ / 'config' / 'cleaning_rules.yaml'

# Reglas de limpieza definidas como diccionario Python
# para evitar errores de sintaxis YAML al escribir el archivo
reglas = {
    'PMC10967698': {
        'remove_patterns': [
            'RSC Advances',
            '## PAPER',
            'rsc.li/rsc-advances',
        ],
        'remove_affiliations': True,
        'fix_ocr_errors': True,
    }
}

# Cabecera de documentación del archivo YAML
cabecera = (
    "# cleaning_rules.yaml\n"
    "# Reglas de limpieza específicas por documento para el pipeline RAG.\n"
    "# La limpieza universal se aplica siempre en 01_ingesta.ipynb.\n"
    "# Añadir una entrada por cada PDF que requiera limpieza adicional.\n"
    "# Clave: nombre del PDF sin extensión.\n\n"
)

with open(RUTA_YAML, 'w', encoding='utf-8') as f:
    f.write(cabecera)
    yaml.dump(
        reglas,
        f,
        default_flow_style=False,
        allow_unicode=True,
        sort_keys=False,
    )

print(f"Archivo generado en: {RUTA_YAML}")

Archivo generado en: /content/drive/MyDrive/chem-rag-assistant/config/cleaning_rules.yaml


## 5. Verificación del archivo YAML generado

In [7]:
# Verificación de que el YAML generado es válido y legible
with open(RUTA_YAML, 'r', encoding='utf-8') as f:
    config_verificada = yaml.safe_load(f)

print("Contenido del archivo YAML:")
print("-" * 45)
print(yaml.dump(config_verificada, default_flow_style=False, allow_unicode=True))
print("-" * 45)
print("Archivo YAML válido y legible")

Contenido del archivo YAML:
---------------------------------------------
PMC10967698:
  fix_ocr_errors: true
  remove_affiliations: true
  remove_patterns:
  - RSC Advances
  - '## PAPER'
  - rsc.li/rsc-advances

---------------------------------------------
Archivo YAML válido y legible


## 6. Verificación de credenciales

In [8]:
# Verificación de que el token de Hugging Face está configurado
# en los secretos de Colab (Settings → Secrets → HF_TOKEN)
import os

if IN_COLAB:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
        if hf_token:
            print("[OK]  HF_TOKEN configurado correctamente en Colab Secrets")
        else:
            print("[AVISO]  HF_TOKEN vacío — configúralo en Settings → Secrets")
    except Exception:
        print("[AVISO]  HF_TOKEN no encontrado — configúralo en Settings → Secrets")
else:
    hf_token = os.getenv('HF_TOKEN')
    if hf_token:
        print("[OK]  HF_TOKEN encontrado en variables de entorno")
    else:
        print("[AVISO]  HF_TOKEN no encontrado — verifica tu archivo .env")

[OK]  HF_TOKEN configurado correctamente en Colab Secrets


## 7. Resumen de la configuración

In [9]:
# Resumen final del estado del entorno
print("=" * 60)
print("RESUMEN — CONFIGURACIÓN DEL ENTORNO")
print("=" * 60)
print(f"  Proyecto raíz     : {PROYECTO_RAIZ}")
print(f"  Carpetas creadas  : {len(CARPETAS)}")
print(f"  Archivo YAML      : {RUTA_YAML.name}")
print(f"  Documentos config : {list(config_verificada.keys())}")
print("=" * 60)
print("Entorno listo — ejecuta 01_ingesta.ipynb para continuar")

RESUMEN — CONFIGURACIÓN DEL ENTORNO
  Proyecto raíz     : /content/drive/MyDrive/chem-rag-assistant
  Carpetas creadas  : 5
  Archivo YAML      : cleaning_rules.yaml
  Documentos config : ['PMC10967698']
Entorno listo — ejecuta 01_ingesta.ipynb para continuar
